In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-gmk90JKmSW4I


In [2]:
# import v1.0
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# LLM 모델
# FewShotPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate

## FewShotPromptTemplate 설계

In [3]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },

    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },

    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },

    {
        "country": "미국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 워싱턴 D.C.
            언어: 영어
            음식: 햄버거와 스테이크
            통화: 달러
        """
    }
]

## 2단계: 예시용 프롬프트 템플릿 정의

In [13]:
example_template = """"
    Human: {country}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='"\n    Human: {country}\n    AI: {answer}\n')

## 3단계: FewShotPromptTemplate 생성 후 결합

In [14]:
prompt = FewShotPromptTemplate(
    # 질문
    example_prompt=example_prompt,
    # 예시
    examples=examples,
    # 사용자의 질문 
    suffix="Human: {country}에 대해서 어떻게 알고 있어요?",
    input_variables=["country"]
)

In [15]:
chat = ChatOpenAI(temperature=0)

In [16]:
chain = prompt | chat

In [17]:
result = chain.invoke({
    "country": "한국"
})

In [19]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원


In [22]:
result = chain.invoke({
    "country": "영국"
})

## FewShotChatPromoptMessage 설계

In [23]:
# v1.0
# ChatModel
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [24]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },

    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },

    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },

    {
        "country": "미국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 워싱턴 D.C.
            언어: 영어
            음식: 햄버거와 스테이크
            통화: 달러
        """
    }
]

## 2단계 예시용 프롬프트 템플릿 정의

In [31]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}")
])

In [32]:
prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_prompt,
    examples = examples
)

In [ ]:
# final = prompt | chain

In [33]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다. 짧은 답변을 제공하세요"),
    prompt,
    ("human", "{country}에대해서 어떻게 알고 있어요?")
])

In [34]:
final_chain = final_prompt | chat

In [35]:
result = final_chain.invoke({
    "country": "일본"
})

## LengthBasedExampleSelector 설계

In [39]:
# v1.0
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [40]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },

    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },

    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },

    {
        "country": "미국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 워싱턴 D.C.
            언어: 영어
            음식: 햄버거와 스테이크
            통화: 달러
        """
    }
]

In [41]:
example_prompt = PromptTemplate.from_template("Human : {country}\nAI: {answer}")

## Selector 연결

In [54]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 예제의 양 허용 (토큰 개수)
    max_length=200
)

In [55]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: {country}에 대해서 어떻게 알고 있나요?",
    input_variables=["country"]
)

In [57]:
print(prompt.format(**{
    "country": "브라질"
}))

Human : 프랑스에 대해서 어떻게 알고 있나요?
AI: 
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        

Human : 대한민국에 대해서 어떻게 알고 있나요?
AI: 
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        

Human: 브라질에 대해서 어떻게 알고 있나요?


In [58]:
final_chain = prompt | chat

In [60]:
result = final_chain.invoke({
    "country": "독일"
}) 

In [61]:
print(result.content)

AI:
            저는 이렇게 알고 있어요.
            수도: 베를린
            언어: 독일어
            음식: 소세지와 맥주
            통화: 유로


## Chat모델(LengthBasedExampleSelector)

In [63]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },

    {
        "country": "대한민국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 원
        """
    },

    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 초밥과 라멘
            통화: 엔
        """
    },

    {
        "country": "미국에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 워싱턴 D.C.
            언어: 영어
            음식: 햄버거와 스테이크
            통화: 달러
        """
    }
]

In [69]:
length_prompt = PromptTemplate(
    input_variables=["country", "answer"],
    template="""
        Human: {country}에 대해서 어떻게 알고 있어요?
        AI: {answer}
"""
)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}")
])

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=length_prompt,
    max_length=200
)

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt
)

In [71]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다."),
    fewshot_prompt,
    ("human", "{country}에 대해 어떻게 알고 있어요?")
])

In [72]:
chain = final_prompt | chat

In [73]:
result = chain.invoke({
    "country": "대한민국"
})

In [75]:
print(result.content)


            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치, 불고기
            통화: 대한민국 원
